# DS 227 · Week 3 — Data scraping I
## Fetching and parsing the portal's calendar page (a saved copy)

Every code block from the lecture, in order, so you can run what you watched instead of
retyping it. Every table is small and made up, typed into the cells (or written to a small file by a setup cell), so there is no file to download.

**How to use this.** Run the cells top to bottom the first time (Runtime → Run all). The
exercise cells list their tasks above the code: predict first, then run, then change one
thing and run again.

**Version differences.** The slides follow pandas 3. On pandas 2, text columns show as
`object` instead of `str`, and dates as `datetime64[ns]` instead of `datetime64[us]`.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

---
## The lecture, cell by cell

Headings carry the slide numbers shown in the deck's counter.


### Slide 23 — A first fetch in three lines


Expected output:

```
200
text/html; charset=utf-8
30586
```


In [ ]:
import requests

URL = ("https://portal.latarak.com/static/datasets/"
       "ds227-calendar-2026-10-22.html")      # a saved copy of /calendar
r = requests.get(URL)

print(r.status_code)
print(r.headers["Content-Type"])
print(len(r.text))      # about 30,000 characters of HTML

### Slide 34 — Text in, tree out


Expected output:

```
Academic Calendar 2026–2027 - Learning Hub
Academic Calendar 2026–2027
```


In [ ]:
from bs4 import BeautifulSoup

soup = BeautifulSoup(r.text, "html.parser")

print(soup.find("title").get_text())
print(soup.find("h1").get_text(strip=True))

### Slide 35 — find gives one; find_all gives a list


Expected output:

```
table
```


In [ ]:
# ONE tag (or None)
table = soup.find("table")
print(table.name)          # a single element: cannot be looped over

### Slide 35 — find gives one; find_all gives a list


Expected output:

```
2
```


In [ ]:
# EVERY match, as a list
tables = soup.find_all("table", class_="cal-table")
print(len(tables))         # the page has 2: loop over them

### Slide 37 — get_text(strip=True) as the default


Setup for the next cell: a name its code uses that the slide defines off-screen.


In [ ]:
cal = soup.find_all("table", class_="cal-table")[0]
headers = [th.get_text(strip=True) for th in cal.find("thead").find_all("th")]
rows = [dict(zip(headers, [td.get_text(strip=True) for td in tr.find_all("td")])) for tr in cal.find("tbody").find_all("tr")]

Expected output:

```
'\nMilestoneDateTerm\n'
'MilestoneDateTerm'
```


In [ ]:
head = cal.find("thead")

print(repr(head.get_text()))
print(repr(head.get_text(strip=True)))

### Slide 38 — .select() uses CSS selectors


Expected output:

```
24
1
```


In [ ]:
# every holiday name on the calendar page
names = soup.select("ul.holiday-list span.h-name")
print(len(names))

# the one main-content element, by id
print(len(soup.select("#main-content")))

# . = class, # = id, space = descendant

### Slide 43 — Fetch, then check before parsing


Expected output:

```
200 30586
```


In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

URL = ("https://portal.latarak.com/static/datasets/"
       "ds227-calendar-2026-10-22.html")      # a saved copy of /calendar
r = requests.get(URL)
r.raise_for_status()   # crashes here if not 2xx

print(r.status_code, len(r.text))

### Slide 44 — Parse, then locate the table


Expected output:

```
2
```


In [ ]:
soup = BeautifulSoup(r.text, "html.parser")

tables = soup.find_all("table", class_="cal-table")
print(len(tables))     # [0] key dates · [1] breaks & notable dates

cal = tables[0]

### Slide 45 — Extract the header row first


Expected output:

```
['Milestone', 'Date', 'Term']
```


In [ ]:
headers = [th.get_text(strip=True)
           for th in cal.find("thead").find_all("th")]

print(headers)

### Slide 46 — Body rows into dicts


Expected output:

```
37
{'Milestone': 'Registration period (First Semester)', 'Date': 'Aug 3 – Aug 7, 2026', 'Term': '1st Sem 2026-2027'}
```


In [ ]:
rows = []
for tr in cal.find("tbody").find_all("tr"):
    cells = [td.get_text(strip=True)
             for td in tr.find_all("td")]
    rows.append(dict(zip(headers, cells)))

print(len(rows))
print(rows[0])

### Slide 48 — Storing the result in pandas


Expected output:

```
(37, 3)
Milestone  ...               Term
0               Registration period (First Semester)  ...  1st Sem 2026-2027
1                  Start of classes (First Semester)  ...  1st Sem 2026-2027
2  Change of matriculation deadline (First Semester)  ...  1st Sem 2026-2027
[3 rows x 3 columns]
```


In [ ]:
df = pd.DataFrame(rows)
print(df.shape)
print(df.head(3))

df.to_csv("up_calendar.csv", index=False)

### Slide 49 — The complete scraper in 14 lines


In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

r = requests.get("https://portal.latarak.com/static/datasets/ds227-calendar-2026-10-22.html")
r.raise_for_status()
soup = BeautifulSoup(r.text, "html.parser")

cal = soup.find_all("table", class_="cal-table")[0]
headers = [th.get_text(strip=True) for th in cal.find("thead").find_all("th")]
rows = [dict(zip(headers, [td.get_text(strip=True) for td in tr.find_all("td")]))
        for tr in cal.find("tbody").find_all("tr")]

df = pd.DataFrame(rows)
df.to_csv("up_calendar.csv", index=False)

### Slide 50 — Reading pure tables with pandas alone


Expected output:

```
2
(37, 3)
```


In [ ]:
from io import StringIO

dfs = pd.read_html(StringIO(r.text))
print(len(dfs))
print(dfs[0].shape)   # the same 37 x 3 table, no BeautifulSoup

### Slide 53 — Handle the optional field explicitly


Expected output:

```
{'date': 'Aug 6', 'note': 'UP Cebu only'}
{'date': 'Aug 21', 'note': None}
```


In [ ]:
hols = []
for li in soup.select("ul.holiday-list")[0].find_all("li"):   # the 2026 list
    small = li.find("small")
    hols.append({
        "date": li.find("span", class_="h-date").get_text(strip=True),
        "note": small.get_text(strip=True) if small else None,
    })

print(hols[0])
print(hols[1])

### Slide 54 — Page changes break scrapers


Setup for the next cell: a name its code uses that the slide defines off-screen.


In [ ]:
holidays = soup.select("ul.holiday-list")[0].find_all("li")

Expected output:

```
UP Cebu only
AttributeError: 'NoneType' object has no attribute 'get_text'
```


This cell **fails on purpose**. It is wrapped in `try`/`except` so the notebook keeps running; the error is printed instead.


In [ ]:
try:
    # fails on purpose: read the last line
    for li in holidays:
        print(li.find("small").get_text())

    # 11 of the 12 holidays have no <small>:
    # the second one stops the loop
except SystemExit as e:
    print(e)
except Exception as e:
    import traceback
    print(traceback.format_exception(e)[-1].strip())

### Slide 56 — Parsing a saved snapshot


Expected output:

```
Academic Calendar 2026–2027
```


In [ ]:
# save once
open("calendar.html", "w").write(r.text)

# parse forever: no network, no drift
html = open("calendar.html").read()
soup = BeautifulSoup(html, "html.parser")
print(soup.find("h1").get_text(strip=True))